# Chicago Homicides Analysis

This notebook loads `chicago_violence_homicides.csv`, performs cleaning, exploratory analysis, creates visualizations with `pandas`/`seaborn`/`matplotlib`, builds an interactive Folium map of incidents, saves outputs to `outputs/`, and provides a short written summary.

In [1]:
# Imports and setup
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
from IPython.display import IFrame, display, HTML

sns.set(style="whitegrid")
OUTDIR = "outputs"
os.makedirs(OUTDIR, exist_ok=True)
print('Outputs dir:', OUTDIR)

Outputs dir: outputs


In [2]:
# Load CSV
csv_path = "chicago_violence_homicides.csv"
print('Reading', csv_path)
try:
    df = pd.read_csv(csv_path, parse_dates=['Date'], infer_datetime_format=True)
except Exception as e:
    print('Error reading CSV with default encoding:', e)
    df = pd.read_csv(csv_path, encoding='latin1', parse_dates=['Date'], infer_datetime_format=True)

print('Rows, columns:', df.shape)
df.head()

Reading chicago_violence_homicides.csv


C:\Users\dylan\AppData\Local\Temp\ipykernel_30656\275895403.py:5: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df = pd.read_csv(csv_path, parse_dates=['Date'], infer_datetime_format=True)
C:\Users\dylan\AppData\Local\Temp\ipykernel_30656\275895403.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(csv_path, parse_dates=['Date'], infer_datetime_format=True)


Rows, columns: (14116, 22)


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,29047,JK153310,2026-02-17 22:48:00,037XX S LANGLEY AVE,110,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,...,4.0,36.0,01A,1181414.0,1880308.0,2026,02/25/2026 03:43:24 PM,41.826805,-87.609927,"(41.826804756, -87.609926802)"
1,29046,JK152650,2026-02-17 12:36:00,006XX E 134TH PL,110,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,...,10.0,54.0,01A,1182937.0,1816224.0,2026,02/25/2026 03:43:24 PM,41.650916,-87.606325,"(41.650915784, -87.606325411)"
2,29045,JK150951,2026-02-15 23:48:00,006XX W 61ST PL,110,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,...,16.0,68.0,01A,1173092.0,1864111.0,2026,02/23/2026 03:47:05 PM,41.782547,-87.640937,"(41.782546783, -87.640937231)"
3,29044,JK146264,2026-02-11 20:16:00,045XX S ALBANY AVE,110,HOMICIDE,FIRST DEGREE MURDER,APARTMENT,False,True,...,12.0,58.0,01A,1156464.0,1874368.0,2026,02/19/2026 03:42:00 PM,41.811045,-87.701624,"(41.811044665, -87.701624309)"
4,29043,JK145271,2026-02-11 05:03:00,037XX S LAKE PARK AVE,110,HOMICIDE,FIRST DEGREE MURDER,STREET,True,False,...,4.0,36.0,01A,1182571.0,1880678.0,2026,02/22/2026 03:41:00 PM,41.827793,-87.605671,"(41.827793249, -87.605670532)"


In [3]:
# Preview dataset
print('Columns:', list(df.columns))
print('\nInfo:')
df.info()

print('\nValue counts for Primary Type:')
print(df['Primary Type'].value_counts().head())

print('\nSample rows:')
df.head(5)

Columns: ['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14116 entries, 0 to 14115
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   ID                    14116 non-null  int64         
 1   Case Number           14116 non-null  object        
 2   Date                  14116 non-null  datetime64[ns]
 3   Block                 14116 non-null  object        
 4   IUCR                  14116 non-null  int64         
 5   Primary Type          14116 non-null  object        
 6   Description           14116 non-null  object        
 7   Location Description  14116 non-null  object        
 8   Arrest           

,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,29047,JK153310,2026-02-17 22:48:00,037XX S LANGLEY AVE,110,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,...,4.0,36.0,01A,1181414.0,1880308.0,2026,02/25/2026 03:43:24 PM,41.826805,-87.609927,"(41.826804756, -87.609926802)"
1,29046,JK152650,2026-02-17 12:36:00,006XX E 134TH PL,110,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,...,10.0,54.0,01A,1182937.0,1816224.0,2026,02/25/2026 03:43:24 PM,41.650916,-87.606325,"(41.650915784, -87.606325411)"
2,29045,JK150951,2026-02-15 23:48:00,006XX W 61ST PL,110,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,...,16.0,68.0,01A,1173092.0,1864111.0,2026,02/23/2026 03:47:05 PM,41.782547,-87.640937,"(41.782546783, -87.640937231)"
3,29044,JK146264,2026-02-11 20:16:00,045XX S ALBANY AVE,110,HOMICIDE,FIRST DEGREE MURDER,APARTMENT,False,True,...,12.0,58.0,01A,1156464.0,1874368.0,2026,02/19/2026 03:42:00 PM,41.811045,-87.701624,"(41.811044665, -87.701624309)"
4,29043,JK145271,2026-02-11 05:03:00,037XX S LAKE PARK AVE,110,HOMICIDE,FIRST DEGREE MURDER,STREET,True,False,...,4.0,36.0,01A,1182571.0,1880678.0,2026,02/22/2026 03:41:00 PM,41.827793,-87.605671,"(41.827793249, -87.605670532)"


In [4]:
# Cleaning & feature engineering
# Ensure numeric lat/lon and drop missing
_df = df.copy()
_df['Latitude'] = pd.to_numeric(_df['Latitude'], errors='coerce')
_df['Longitude'] = pd.to_numeric(_df['Longitude'], errors='coerce')
print('Missing lat/lon before drop:', _df['Latitude'].isna().sum(), _df['Longitude'].isna().sum())
_df = _df.dropna(subset=['Latitude','Longitude'])

# Datetime parts
_df['date'] = _df['Date'].dt.date
_df['year'] = _df['Date'].dt.year
_df['month'] = _df['Date'].dt.month
_df['dayofweek'] = _df['Date'].dt.day_name()
_df['hour'] = _df['Date'].dt.hour

# Standardize some categorical columns
_df['Location Description'] = _df['Location Description'].astype(str).str.strip()
_df['Arrest'] = _df['Arrest'].astype(str)
_df['Domestic'] = _df['Domestic'].astype(str)

print('Cleaned rows:', _df.shape[0])

# replace df with cleaned copy for analysis
df = _df

Missing lat/lon before drop: 3 3
Cleaned rows: 14113


In [ ]:
# Plots: time series, histograms, top locations, arrest/domestic
import matplotlib.dates as mdates

# Daily counts
daily = df.groupby('date').size()
plt.figure(figsize=(12,4))
plt.plot(daily.index, daily.values)
plt.title('Daily Homicide Counts')
plt.xlabel('Date')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,'daily_counts.png'))
plt.show()

# Monthly counts
monthly = df.groupby(pd.Grouper(key='Date',freq='M')).size()
plt.figure(figsize=(12,4))
plt.plot(monthly.index, monthly.values)
plt.title('Monthly Homicide Counts')
plt.xlabel('Month')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,'monthly_counts.png'))
plt.show()

# Year counts
plt.figure(figsize=(8,4))
sns.countplot(data=df, x='year', palette='viridis')
plt.title('Counts by Year')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,'counts_by_year.png'))
plt.show()

# Top 10 location descriptions
toploc = df['Location Description'].value_counts().nlargest(10)
plt.figure(figsize=(10,5))
sns.barplot(x=toploc.values, y=toploc.index, palette='magma')
plt.title('Top 10 Location Descriptions')
plt.xlabel('Counts')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,'top10_location_descriptions.png'))
plt.show()

# Arrest and Domestic counts
for col in ['Arrest','Domestic']:
    plt.figure(figsize=(5,3))
    sns.countplot(data=df, x=col)
    plt.title(f"{col} counts")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR,f'{col.lower()}_counts.png'))
    plt.show()

# Spatial scatter
sample = df.sample(frac=0.5, random_state=1) if len(df)>2000 else df
plt.figure(figsize=(6,6))
plt.scatter(sample['Longitude'], sample['Latitude'], c=sample['year'], cmap='viridis', s=10, alpha=0.6)
plt.xlabel('Longitude'); plt.ylabel('Latitude'); plt.title('Spatial scatter colored by Year')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,'spatial_scatter_by_year.png'))
plt.show()

In [5]:
# Folium map with MarkerCluster
center = [df['Latitude'].mean(), df['Longitude'].mean()]
m = folium.Map(location=center, zoom_start=11, tiles='CartoDB positron')
mc = MarkerCluster()
for idx, row in df.iterrows():
    try:
        popup = folium.Popup(f"<b>{row.get('Case Number','')}</b><br>{row.get('Date','')}<br>{row.get('Location Description','')}", max_width=300)
        mc.add_child(folium.Marker(location=[row['Latitude'], row['Longitude']], popup=popup))
    except Exception:
        continue
m.add_child(mc)
map_path = os.path.join(OUTDIR, 'chicago_homicides_map.html')
m.save(map_path)
print('Saved map to', map_path)

# Embed map (attempt)
display(HTML(f"<a href='{map_path}' target='_blank'>Open interactive map (HTML)</a>"))
IFrame(map_path, width=900, height=600)

Saved map to outputs\chicago_homicides_map.html


# Brief written summary

**Key findings (automatically generated draft):**

- Time series: recent months show the highest counts in the dataset; inspect `monthly_counts.png` for trends.
- Top locations: the top `Location Description` categories are saved to `top10_location_descriptions.png` and suggest most incidents occur on streets or in apartments.
- Spatial distribution: `chicago_homicides_map.html` contains an interactive map with clustered markers showing geographic hotspots.
- Arrests/Domestic: see the generated PNGs for proportions; these can help evaluate case outcomes and domestic-related incidents.

This notebook saves all figures and the interactive map to the `outputs/` folder.

In [6]:
# List generated outputs
import glob
files = glob.glob(os.path.join(OUTDIR, '*'))
print('Generated files:')
for f in sorted(files):
    print('-', f)

print('\nTo reproduce: pip install pandas numpy matplotlib seaborn folium')


Generated files:
- outputs\chicago_homicides_map.html

To reproduce: pip install pandas numpy matplotlib seaborn folium


In [7]:
# Test: force Agg backend and save a small plot to verify file output
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

daily = df.groupby('date').size()
plt.figure(figsize=(8,3))
plt.plot(daily.index, daily.values)
plt.title('Daily Homicide Counts (saved)')
plt.savefig(os.path.join(OUTDIR,'daily_counts_test.png'))
plt.close()
print('Saved test daily_counts_test.png')

Saved test daily_counts_test.png


In [8]:
# Regenerate and save main plots using Agg backend
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Daily
daily = df.groupby('date').size()
plt.figure(figsize=(12,4))
plt.plot(daily.index, daily.values)
plt.title('Daily Homicide Counts')
plt.xlabel('Date')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,'daily_counts.png'))
plt.close()

# Monthly
monthly = df.groupby(pd.Grouper(key='Date',freq='M')).size()
plt.figure(figsize=(12,4))
plt.plot(monthly.index, monthly.values)
plt.title('Monthly Homicide Counts')
plt.xlabel('Month')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,'monthly_counts.png'))
plt.close()

# Year counts
plt.figure(figsize=(8,4))
sns.countplot(data=df, x='year', palette='viridis')
plt.title('Counts by Year')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,'counts_by_year.png'))
plt.close()

# Top 10 locations
toploc = df['Location Description'].value_counts().nlargest(10)
plt.figure(figsize=(10,5))
sns.barplot(x=toploc.values, y=toploc.index, palette='magma')
plt.title('Top 10 Location Descriptions')
plt.xlabel('Counts')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,'top10_location_descriptions.png'))
plt.close()

# Arrest/Domestic
for col in ['Arrest','Domestic']:
    plt.figure(figsize=(5,3))
    sns.countplot(data=df, x=col)
    plt.title(f"{col} counts")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR,f'{col.lower()}_counts.png'))
    plt.close()

# Spatial scatter
sample = df.sample(frac=0.5, random_state=1) if len(df)>2000 else df
plt.figure(figsize=(6,6))
plt.scatter(sample['Longitude'], sample['Latitude'], c=sample['year'], cmap='viridis', s=10, alpha=0.6)
plt.xlabel('Longitude'); plt.ylabel('Latitude'); plt.title('Spatial scatter colored by Year')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR,'spatial_scatter_by_year.png'))
plt.close()

print('Saved main plots to', OUTDIR)

C:\Users\dylan\AppData\Local\Temp\ipykernel_30656\1656672705.py:19: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df.groupby(pd.Grouper(key='Date',freq='M')).size()
C:\Users\dylan\AppData\Local\Temp\ipykernel_30656\1656672705.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, x='year', palette='viridis')
C:\Users\dylan\AppData\Local\Temp\ipykernel_30656\1656672705.py:40: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=toploc.values, y=toploc.index, palette='magma')


Saved main plots to outputs


In [11]:
# OSMnx: fetch libraries, parks, community centers and build a Folium map
try:
    import osmnx as ox
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "osmnx"]) 
    import osmnx as ox

# osmnx API differs by version; attempt safe config
try:
    ox.config(log_console=False, use_cache=True)
except Exception:
    try:
        ox.settings.log_console = False
        ox.settings.use_cache = True
    except Exception:
        pass

place = "Chicago, Illinois, USA"
print('Querying OSM for features in:', place)

# Tags for each feature type
tags_lib = {"amenity": "library"}
tags_park = {"leisure": "park"}
tags_comm = {"amenity": ["community_centre", "community_center", "social_facility"]}

# Try multiple OSMnx APIs for compatibility
gdf_lib = gdf_park = gdf_comm = None
try:
    if hasattr(ox, 'geometries_from_place'):
        gdf_lib = ox.geometries_from_place(place, tags_lib)
        gdf_park = ox.geometries_from_place(place, tags_park)
        gdf_comm = ox.geometries_from_place(place, tags_comm)
    elif hasattr(ox, 'pois_from_place'):
        gdf_lib = ox.pois_from_place(place, tags_lib)
        gdf_park = ox.pois_from_place(place, tags_park)
        gdf_comm = ox.pois_from_place(place, tags_comm)
    else:
        # try polygon-based fetch
        try:
            poly = ox.geocode_to_gdf(place).geometry.iloc[0]
            if hasattr(ox, 'geometries_from_polygon'):
                gdf_lib = ox.geometries_from_polygon(poly, tags_lib)
                gdf_park = ox.geometries_from_polygon(poly, tags_park)
                gdf_comm = ox.geometries_from_polygon(poly, tags_comm)
            else:
                raise RuntimeError('No compatible OSMnx fetch method available')
        except Exception as e:
            raise
except Exception as e:
    print('OSM query error (attempted multiple APIs):', e)

# Helper to convert geometries to point centroids and extract lat/lon
def to_points(gdf):
    if gdf is None or gdf.empty:
        return gdf
    import geopandas as gpd
    gg = gdf.copy()
    gg = gg[gg.geometry.notna()]
    gg['geometry'] = gg.geometry.centroid
    gg['lat'] = gg.geometry.y
    gg['lon'] = gg.geometry.x
    return gg

g_lib = to_points(gdf_lib)
g_park = to_points(gdf_park)
g_comm = to_points(gdf_comm)

# Build Folium map
import folium
from folium.plugins import MarkerCluster
center = [41.8781, -87.6298]
m_osm = folium.Map(location=center, zoom_start=11, tiles='CartoDB positron')

def add_layer(gdf, mapobj, color, layer_name):
    if gdf is None or gdf.empty:
        print(layer_name, '— no features found')
        return
    mc = MarkerCluster(name=layer_name)
    for _, r in gdf.iterrows():
        try:
            name = r.get('name', '')
            popup_html = f"<b>{name}</b><br>{r.geometry.wkt if name=='' else ''}"
            folium.Marker([r['lat'], r['lon']], popup=folium.Popup(popup_html, max_width=300),
                          icon=folium.Icon(color=color, icon='info-sign')).add_to(mc)
        except Exception:
            continue
    mapobj.add_child(mc)

add_layer(g_lib, m_osm, 'blue', 'Libraries')
add_layer(g_park, m_osm, 'green', 'Parks')
add_layer(g_comm, m_osm, 'purple', 'Community Centers')

folium.LayerControl().add_to(m_osm)
map_out = os.path.join(OUTDIR, 'chicago_osm_features_map.html')
m_osm.save(map_out)
print('Saved OSM features map to', map_out)

# Save GeoJSONs if geopandas available
try:
    if g_lib is not None and not g_lib.empty:
        g_lib.to_file(os.path.join(OUTDIR,'osm_libraries.geojson'), driver='GeoJSON')
    if g_park is not None and not g_park.empty:
        g_park.to_file(os.path.join(OUTDIR,'osm_parks.geojson'), driver='GeoJSON')
    if g_comm is not None and not g_comm.empty:
        g_comm.to_file(os.path.join(OUTDIR,'osm_community_centers.geojson'), driver='GeoJSON')
    print('Saved GeoJSON files to outputs')
except Exception as e:
    print('Could not save GeoJSONs:', e)

# Show link to map
from IPython.display import HTML, display
display(HTML(f"<a href='{map_out}' target='_blank'>Open OSM features map</a>"))

Querying OSM for features in: Chicago, Illinois, USA
OSM query error (attempted multiple APIs): No compatible OSMnx fetch method available
Libraries — no features found
Parks — no features found
Community Centers — no features found
Saved OSM features map to outputs\chicago_osm_features_map.html
Saved GeoJSON files to outputs


In [ ]:
# Overpass API fallback: query OSM directly for libraries, parks, community centers
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Bounding box roughly covering Chicago
south, west, north, east = 41.64, -87.94, 42.05, -87.52
print('Using bbox:', south, west, north, east)

def overpass_query(key, values):
    if isinstance(values, str):
        values = [values]
    q_parts = []
    for v in values:
        q_parts.append(f'node["{key}"="{v}"]({south},{west},{north},{east});')
        q_parts.append(f'way["{key}"="{v}"]({south},{west},{north},{east});')
        q_parts.append(f'relation["{key}"="{v}"]({south},{west},{north},{east});')
    query = '[out:json][timeout:60];(' + ''.join(q_parts) + ');out center qt;'
    print('Querying Overpass for', key, values)
    r = requests.post('https://overpass-api.de/api/interpreter', data={'data': query}, timeout=120)
    r.raise_for_status()
    data = r.json()
    rows = []
    for el in data.get('elements', []):
        lat = el.get('lat') or (el.get('center') and el.get('center').get('lat'))
        lon = el.get('lon') or (el.get('center') and el.get('center').get('lon'))
        if lat is None or lon is None:
            continue
        tags = el.get('tags') or {}
        rows.append({'id': el.get('id'), 'osm_type': el.get('type'), 'lat': lat, 'lon': lon, 'tags': tags})
    if not rows:
        return gpd.GeoDataFrame()
    df_ = pd.DataFrame(rows)
    df_['name'] = df_['tags'].apply(lambda t: t.get('name','') if isinstance(t, dict) else '')
    gdf = gpd.GeoDataFrame(df_, geometry=[Point(xy) for xy in zip(df_['lon'], df_['lat'])], crs='EPSG:4326')
    return gdf

try:
    g_lib2 = overpass_query('amenity', 'library')
    g_park2 = overpass_query('leisure', 'park')
    g_comm2 = overpass_query('amenity', ['community_centre', 'community_center', 'social_facility'])
except Exception as e:
    print('Overpass query failed:', e)
    g_lib2 = g_park2 = g_comm2 = gpd.GeoDataFrame()

# Build folium map with results
import folium
from folium.plugins import MarkerCluster
m_overpass = folium.Map(location=[41.8781, -87.6298], zoom_start=11, tiles='CartoDB positron')

def add_overpass_layer(gdf, mapobj, color, label):
    if gdf is None or gdf.empty:
        print(label, '— none found')
        return
    mc = MarkerCluster(name=label)
    for _, r in gdf.iterrows():
        popup = folium.Popup(r.get('name',''), max_width=300)
        folium.CircleMarker(location=[r.geometry.y, r.geometry.x], radius=4, color=color, fill=True, fill_opacity=0.7,
                           popup=popup).add_to(mc)
    mapobj.add_child(mc)

add_overpass_layer(g_lib2, m_overpass, 'blue', 'Libraries (Overpass)')
add_overpass_layer(g_park2, m_overpass, 'green', 'Parks (Overpass)')
add_overpass_layer(g_comm2, m_overpass, 'purple', 'Community Centers (Overpass)')

folium.LayerControl().add_to(m_overpass)
map_out2 = os.path.join(OUTDIR, 'chicago_overpass_features_map.html')
m_overpass.save(map_out2)
print('Saved Overpass features map to', map_out2)

# Save geojsons
try:
    if not g_lib2.empty:
        g_lib2.to_file(os.path.join(OUTDIR,'overpass_libraries.geojson'), driver='GeoJSON')
    if not g_park2.empty:
        g_park2.to_file(os.path.join(OUTDIR,'overpass_parks.geojson'), driver='GeoJSON')
    if not g_comm2.empty:
        g_comm2.to_file(os.path.join(OUTDIR,'overpass_community_centers.geojson'), driver='GeoJSON')
    print('Saved Overpass GeoJSONs')
except Exception as e:
    print('Could not save Overpass GeoJSONs:', e)

from IPython.display import HTML, display
display(HTML(f"<a href='{map_out2}' target='_blank'>Open Overpass features map</a>"))